# EPOCH 5th Datathon 본선 — Team3 빡빡이 최종 제출 코드

**최종 제출 = FINAL21**: `try_shot2` 블렌드(mate 0.70 + T01 0.20 + q_joint 0.10, 순위 블렌딩) 위에
**"지난주(08-25) 정답 1 + 08-25~08-31 롱폼 업로드 0편"** 채널을 최상위로 올리는 규칙 1개를 적용.

| 구성 | 설명 | 단독 LB |
|---|---|---|
| mate | 팀원 구조 시뮬레이션(sim.py) 0.8 + T01 0.2 | 0.7395 |
| T01 | ranklearn 모델 | 0.7032 |
| q_joint | 로지스틱 + 순위변환 + 우측 중도절단 소프트 라벨 (`run_final.py --lag1feat --lag1q --q_joint`) | 0.7115 |
| blend_90 | mate 0.9 + q_joint 0.1 | **0.7412** |

**규칙 근거 (테스트와 같은 구조 백테스트: 다음 창 26일, 직전 라벨 = 7일 전 18일 창, 07-25~08-05의 12개 기준일)**
- 지난주 정답 1인데 관측 구간(직전 7일)에 롱폼 0편 → 그 정답은 기준일 이후 영상으로 결정됨 → 그 영상들은 테스트 창 안에 포함
- 해당 그룹의 다음 정답 양성률 **77.5%** (31/40), 전체 평균 19%, 지난주 정답 1 전체 50%
- 시뮬레이션 점수에 같은 규칙 적용 시 AP +0.007 (12일 중 8일 개선)

입력: `./epoch_data/`(sample_submission, train_labels, video_5d_views) + 모델 출력 `mate.csv`, `blends/T01_ranklearn50_S01x50.csv`, `submit_qjoint.csv`
랜덤 요소 없음(결정적). `pandas`, `numpy`만 사용.

In [ ]:
import os, numpy as np, pandas as pd
DATA = './epoch_data'
MATE, T01, QJ = 'mate.csv', 'blends/T01_ranklearn50_S01x50.csv', 'submit_qjoint.csv'
D_TEST, D_PREV = '2026-09-01', '2026-08-25'
def clean(df):
    df.columns = [c.strip().lstrip('\ufeff') for c in df.columns]; return df
ss = clean(pd.read_csv(os.path.join(DATA, 'sample_submission.csv')))
order = ss.row_id.astype(str).tolist()
print('rows', len(order))

## 1. try_shot2 블렌드 (blend_t01.py 와 동일 공식)
`rank = rankpct(Σ w_i · rankpct(src_i))`, `pred = 1/(1+exp(-8(rank-0.8)))`

In [ ]:
def rankpct(path):
    s = clean(pd.read_csv(path)).set_index('row_id').prediction.reindex(order)
    s = s.fillna(s.min())
    return s.rank(pct=True).values
W = {'mate': 0.70, 't01': 0.20, 'qj': 0.10}
score = W['mate']*rankpct(MATE) + W['t01']*rankpct(T01) + W['qj']*rankpct(QJ)
rr = pd.Series(score).rank(pct=True).values
pred = 1/(1+np.exp(-8*(rr-0.8)))
sub = pd.DataFrame({'row_id': order, 'prediction': np.round(pred, 8)})
sub.head()

## 2. 규칙: 지난주 정답 1 + 관측 구간(08-25~08-31) 롱폼 0편 → 최상위 (원래 순서 유지)

In [ ]:
v = clean(pd.read_csv(os.path.join(DATA, 'video_5d_views.csv')))
v['published_at'] = pd.to_datetime(v.published_at, utc=True).dt.tz_localize(None)
L = v[(v.is_shorts == 0) & v.view_5d.notna()]
n_obs = L[(L.published_at >= D_PREV) & (L.published_at < D_TEST)].groupby('channel_id').size()
tl = clean(pd.read_csv(os.path.join(DATA, 'train_labels.csv')))
tl = tl[tl.row_id.astype(str).str.endswith(D_PREV)]
prev = pd.Series(tl.target.values, index=tl.row_id.str[:-len(D_PREV)-1])
ch = sub.row_id.str[:-len(D_TEST)-1]
A = ch.map(prev).eq(1) & ch.map(n_obs).fillna(0).eq(0)
print('규칙 대상 채널 수:', int(A.sum()))   # 9
r = sub.prediction.rank(ascending=False, method='first')
mx = sub.prediction.max(); ra = r[A]
sub.loc[A, 'prediction'] = np.minimum(0.99999, mx + (1-mx)*0.5*(1-(ra-ra.min())/(ra.max()-ra.min()+1)*0.5))
print('대상 채널 새 순위:', sorted(sub.prediction.rank(ascending=False, method='first')[A].astype(int)))

## 3. 저장 및 형식 점검

In [ ]:
OUT = '5th_datathon_submission_빡빡이.csv'
sub.to_csv(OUT, index=False, encoding='utf-8', lineterminator='\n')
chk = pd.read_csv(OUT)
assert len(chk) == len(order) and list(chk.columns) == ['row_id', 'prediction']
assert chk.prediction.between(0, 1).all() and chk.prediction.notna().all() and not chk.row_id.duplicated().any()
print('OK', OUT, len(chk))

## 부록: 구성 모델 재현
- **q_joint**: `python run_final.py --data ./epoch_data --F 26 --prev_F 18 --lag1feat --lag1q --q_joint --mode lb7627 --q_x --soft_k --out a.csv` 와 `--mode latest --out b.csv` 의 순위 반반 (`pct=True`)
- **sim (mate의 80%)**: `python run_sim.py --S 8000 --seed 0` → `sim_p.csv`
- **mate** = 0.8·rank(sim) + 0.2·rank(T01)
- 검증: 시간 기반(데이터가 검증일에 끝난다고 놓고 소프트 라벨까지 재생성). KFold 사용 안 함.